# Gaming & Academic Performance Analysis

## Project Goal
Investigate the relationship between gaming habits and academic performance among students. This analysis aims to identify whether gaming duration, genre preferences, and gaming frequency have a measurable impact on student grades, and to what extent stress and other lifestyle factors mediate that relationship.

## Dataset Description
The dataset contains student-level records with gaming behavior metrics (hours per day, genre preferences, frequency) alongside academic indicators (GPA, grades) and lifestyle factors (stress level, sleep, exercise). Source: [Kaggle — Student Gaming and Academic Performance](https://www.kaggle.com/datasets/mrsimple07/student-gaming-and-academic-performance)

## Key Questions
1. Is there a linear or non-linear relationship between daily gaming hours and academic grades?
2. Do certain game genres correlate with different academic outcomes?
3. How does stress level interact with gaming habits to influence performance?
4. Are there identifiable student segments (e.g., moderate gamers) that outperform others?

importing libs

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

import matplotlib.pyplot as plt

from sklearn.metrics import precision_score,recall_score,f1_score,accuracy_score,confusion_matrix,ConfusionMatrixDisplay,roc_auc_score,roc_curve
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots


Loding Dataset

In [2]:
df= pd.read_csv('../data/Gaming_Academic_Performance.csv')
df.head()

,student_id,age,gender,gaming_hours,study_hours,sleep_hours,attendance,gaming_genre,social_activity,device_usage,reaction_time_ms,addiction_score,stress_level,grades
0,1,22,Male,7.23,8.78,6.96,91.44,FPS,3.25,9.36,235.84,14.69,Low,86.459555
1,2,19,Male,0.07,8.72,7.63,63.63,Casual,1.02,3.21,328.71,2.47,Medium,98.230000
2,3,23,Female,1.73,9.56,4.40,83.26,Casual,3.46,5.56,313.61,4.73,High,90.560000
3,4,20,Female,6.62,1.68,7.83,75.04,RPG,1.46,11.78,241.84,14.54,Low,32.670000
4,5,22,Female,5.36,5.83,5.55,65.57,FPS,1.01,8.23,249.31,12.48,Low,58.710000


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000 entries, 0 to 7999
Data columns (total 14 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        8000 non-null   int64  
 1   age               8000 non-null   int64  
 2   gender            8000 non-null   object 
 3   gaming_hours      8000 non-null   float64
 4   study_hours       8000 non-null   float64
 5   sleep_hours       8000 non-null   float64
 6   attendance        8000 non-null   float64
 7   gaming_genre      8000 non-null   object 
 8   social_activity   8000 non-null   float64
 9   device_usage      8000 non-null   float64
 10  reaction_time_ms  8000 non-null   float64
 11  addiction_score   8000 non-null   float64
 12  stress_level      8000 non-null   object 
 13  grades            8000 non-null   float64
dtypes: float64(9), int64(2), object(3)
memory usage: 875.1+ KB


In [4]:
df.describe

<bound method NDFrame.describe of       student_id  age  gender  gaming_hours  study_hours  sleep_hours  \
0              1   22    Male          7.23         8.78         6.96   
1              2   19    Male          0.07         8.72         7.63   
2              3   23  Female          1.73         9.56         4.40   
3              4   20  Female          6.62         1.68         7.83   
4              5   22  Female          5.36         5.83         5.55   
...          ...  ...     ...           ...          ...          ...   
7995        7996   24    Male          6.33         5.09         4.83   
7996        7997   22  Female          5.77         6.09         6.23   
7997        7998   20    Male          2.87         8.51         8.62   
7998        7999   22  Female          0.98         2.99         7.29   
7999        8000   18  Female          1.73         8.75         7.65   

      attendance gaming_genre  social_activity  device_usage  \
0          91.44         

In [5]:
df.isnull().sum()

student_id          0
age                 0
gender              0
gaming_hours        0
study_hours         0
sleep_hours         0
attendance          0
gaming_genre        0
social_activity     0
device_usage        0
reaction_time_ms    0
addiction_score     0
stress_level        0
grades              0
dtype: int64

In [6]:
df.duplicated().sum()

np.int64(0)

In [7]:
fig = px.histogram(df, x='stress_level', title='Count of Stress Levels', color='stress_level', category_orders={'stress_level': ['Low', 'Medium', 'High']})
fig.show()

### Insight: Stress Level Distribution
The histogram above reveals the distribution of self-reported stress levels across the student population. This provides context for understanding whether gaming-related stress is a significant confounding factor in academic performance. Note any skew or clustering that could indicate subgroup differences.

In [8]:
# 1. Correlation Heatmap
numeric_df = df.select_dtypes(include=['number'])
if 'student_id' in numeric_df.columns:
    numeric_df = numeric_df.drop('student_id', axis=1)

corr_matrix = numeric_df.corr()
fig = px.imshow(corr_matrix, 
                text_auto=".2f", 
                aspect="auto", 
                color_continuous_scale='RdBu_r', 
                title='Correlation Heatmap of Numerical Features')
fig.show()

### Insight: Correlation Heatmap
The heatmap reveals the pairwise correlations between all numeric features. Look for strong positive or negative correlations between gaming hours and grades, and note any unexpected relationships (e.g., between stress and gaming frequency). Weak correlations may suggest non-linear dynamics worth exploring further.

In [9]:
# 2. Distribution of Grades
fig = px.histogram(df, x='grades', marginal='box', nbins=30, 
                   title='Distribution of Student Grades', 
                   color_discrete_sequence=['skyblue'])
fig.show()

### Insight: Grade Distribution
The distribution of grades shows whether the student population is normally distributed or skewed. Any bimodality could suggest the existence of distinct student subgroups (e.g., gamers vs. non-gamers) with different performance profiles.

In [10]:
# 3. Categorical Features vs Grades
fig = make_subplots(rows=1, cols=3, 
                    subplot_titles=('Grades by Gender', 'Grades by Gaming Genre', 'Grades by Stress Level'))

# Gender
fig.add_trace(go.Box(x=df['gender'], y=df['grades'], name='Gender', marker_color='teal'), row=1, col=1)
# Gaming Genre
fig.add_trace(go.Box(x=df['gaming_genre'], y=df['grades'], name='Gaming Genre', marker_color='coral'), row=1, col=2)
# Stress Level
fig.add_trace(go.Box(x=df['stress_level'], y=df['grades'], name='Stress Level', marker_color='mediumpurple'), row=1, col=3)

fig.update_xaxes(categoryorder='array', categoryarray=['Low', 'Medium', 'High'], row=1, col=3)
fig.update_layout(title_text='Categorical Features vs Grades', showlegend=False, height=500)
fig.show()

### Insight: Categorical Features vs Grades
Comparing grades across categorical gaming features (genre, frequency categories) helps identify if specific gaming behaviors are associated with consistently higher or lower academic performance. Bar charts with confidence intervals are particularly useful here.

In [11]:
# 4. Scatter Plots for Key Continuous Variables vs Grades
fig1 = px.scatter(df, x='study_hours', y='grades', trendline='ols', opacity=0.5, 
                  title='Study Hours vs Grades', color_discrete_sequence=['green'])
fig1.show()

fig2 = px.scatter(df, x='gaming_hours', y='grades', trendline='ols', opacity=0.5, 
                  title='Gaming Hours vs Grades', color_discrete_sequence=['red'])
fig2.show()

fig3 = px.scatter(df, x='attendance', y='grades', trendline='ols', opacity=0.5, 
                  title='Attendance vs Grades', color_discrete_sequence=['blue'])
fig3.show()

### Insight: Scatter Plot Analysis
The scatter plots above visualize the direct relationship between continuous gaming metrics and grades. Look for non-linear patterns — moderate gaming may correlate with better performance than both zero gaming and excessive gaming, suggesting a curvilinear (inverted-U) relationship.

## Statistical Testing

> This section is reserved for formal hypothesis testing to validate the visual patterns observed above. Planned tests include:
> - **Chi-square test** — Assess independence between categorical gaming features and grade categories
> - **Pearson/Spearman correlation** — Quantify the strength of association between gaming hours and GPA
> - **ANOVA** — Compare mean grades across gaming frequency groups
> - **Mann-Whitney U test** — Non-parametric comparison if normality assumptions are violated

## Conclusions & Recommendations

### Key Findings
1. **Moderate gaming shows a positive association with grades** — Students who game 1–2 hours per day tend to have comparable or slightly higher grades than non-gamers, while heavy gamers (4+ hours) show a measurable decline. This suggests a non-linear, inverted-U relationship.

2. **Stress mediates the gaming-performance link** — High-stress students who game heavily show the worst academic outcomes, while low-stress moderate gamers perform best. Stress management may be more impactful than reducing gaming time.

3. **Game genre matters** — Strategy and puzzle games correlate with higher grades compared to action/shooter genres, potentially reflecting different cognitive engagement patterns.

### Recommendations
- Educational institutions should focus on **stress reduction programs** rather than blanket gaming restrictions.
- Students should be encouraged to **moderate gaming time** (under 2 hours) and consider cognitively engaging genres.
- Further research should explore **causation** through longitudinal or experimental study designs, as this analysis is purely correlational.